# Lab 5 — Word2Vec, FastText, Doc2Vec, and Clustering (Tamil)

Uses the **same 1,000,000-sentence Tamil training corpus and the same
dev/test split** as Lab 3 / Lab 4 (`ai4bharat/IndicCorpV2`), following the
gensim tutorials linked in the assignment:

- [Word2Vec tutorial](https://radimrehurek.com/gensim/auto_examples/tutorials/run_word2vec.html)
- [FastText tutorial](https://radimrehurek.com/gensim/auto_examples/tutorials/run_fasttext.html)
- [Doc2Vec tutorial](https://radimrehurek.com/gensim/auto_examples/tutorials/run_doc2vec_lee.html)

**Tasks covered:**
1. Train Word2Vec, FastText, and Doc2Vec on the training data.
2. K-means cluster the word vectors; represent each cluster by its 20
   nearest words to the centroid.
3. Represent every dev/test sentence as a vector (via averaged word
   embeddings, and separately via Doc2Vec), and for each sentence find its
   closest sentence.

## Two design decisions the assignment leaves implicit — stated up front

- **"For each sentence in these sets, find the closest sentence."** The task
  only asks you to vectorize the *validation and test* sets — it never asks
  you to vectorize the training set. The plain reading is therefore: for
  each dev sentence, find its nearest neighbor **among the other dev
  sentences** (and likewise test-within-test), not a lookup against the
  1,000,000-sentence training corpus. That's what's implemented below. It's
  also computationally trivial either way (1000×1000 similarities), so the
  choice is about matching the assignment's wording, not about performance.
- **"word embeddings" (task 2b)**: interpreted as the two methods the
  assignment explicitly trains for this purpose — **averaged Word2Vec**
  vectors and **Doc2Vec** inferred vectors. Averaged **FastText** vectors are
  included as a third, bonus column, since FastText is trained anyway and
  its subword-based OOV handling is a genuinely useful contrast to Word2Vec.

## Reusing Lab 3/4's split (same leakage fix as Lab 4)

`dev.txt` and `test.txt` are copied verbatim from Lab 4 — never regenerated.
The training corpus is either a cached `data/train_1000000.txt` (if you
already produced one running Lab 4 at full scale) or streamed fresh with
Lab 4's exact method, **excluding every dev/test sentence**, with the same
leakage-verification cell as Lab 4.

## `DEBUG` mode (same convention as Lab 3/4)

Ships with **`DEBUG = True`**, training on Lab 3/4's 12,000-sentence debug
subset so every cell here is verified end-to-end on real Tamil text with
sane-looking nearest-neighbor words (checked explicitly in §5). **Set
`DEBUG = False`** (on Colab, per your plan) to train on the real
1,000,000-sentence corpus.


In [1]:
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                 "gensim", "scikit-learn", "numpy", "pandas", "matplotlib",
                 "requests", "tqdm", "huggingface_hub"])


CompletedProcess(args=['/home/deeraj/miniconda3/bin/python3.13', '-m', 'pip', 'install', '-q', 'gensim', 'scikit-learn', 'numpy', 'pandas', 'matplotlib', 'requests', 'tqdm', 'huggingface_hub'], returncode=0)

In [2]:
import os
import random
import time
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

from gensim.models import Word2Vec, FastText
from gensim.models.doc2vec import Doc2Vec, TaggedDocument
from sklearn.cluster import KMeans

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

DATA_DIR = "data"
os.makedirs(DATA_DIR, exist_ok=True)


## 1. Configuration

In [3]:
LANG = "ta"
DEBUG = True  # <-- set to False (on Colab) for the full 1,000,000-sentence run

TARGET_TRAIN_SIZE = 1_000_000
DEV_PATH = os.path.join(DATA_DIR, "dev.txt")
TEST_PATH = os.path.join(DATA_DIR, "test.txt")
DEBUG_TRAIN_PATH = os.path.join(DATA_DIR, "train_debug_12000.txt")
FULL_TRAIN_PATH = os.path.join(DATA_DIR, f"train_{TARGET_TRAIN_SIZE}.txt")

# embedding hyperparameters -- shared across Word2Vec / FastText / Doc2Vec
# so the three are as comparable as the algorithms allow
VECTOR_SIZE = 100
WINDOW = 5
MIN_COUNT = 5      # gensim's own default; independent of Lab 4's MIN_COUNT=2
EPOCHS = 10
WORKERS = 1        # single-threaded on purpose: gensim's async SGD is not
                    # bit-reproducible across threads even with a fixed seed;
                    # workers=1 trades training speed for a reproducible run
                    # (raise this on Colab if you don't need exact repro)

# K-means
TOP_N_WORDS_FOR_CLUSTERING = 300 if DEBUG else 20_000
N_CLUSTERS = 15 if DEBUG else 50
WORDS_PER_CLUSTER = 20

print(f"DEBUG={DEBUG}  TOP_N_WORDS_FOR_CLUSTERING={TOP_N_WORDS_FOR_CLUSTERING}  N_CLUSTERS={N_CLUSTERS}")


DEBUG=True  TOP_N_WORDS_FOR_CLUSTERING=300  N_CLUSTERS=15


## 2. Load dev/test (verbatim) and the training corpus (cached-or-streamed)

`stream_training_corpus` is the identical function used in Lab 4: random
byte-offset start, HTTP range request against the 11.8 GB IndicCorp v2 file,
Indic NLP Library sentence splitting, cleaning, and — the important part —
skipping any sentence that matches `dev.txt` or `test.txt` so the training
set streamed here cannot leak into evaluation.


In [4]:
assert os.path.exists(DEV_PATH) and os.path.exists(TEST_PATH), (
    "Copy Lab4/data/dev.txt and Lab4/data/test.txt into this notebook's data/ "
    "folder before running (on Colab: upload them alongside this notebook).")

dev_sentences = [l.rstrip("\n") for l in open(DEV_PATH, encoding="utf-8")]
test_sentences = [l.rstrip("\n") for l in open(TEST_PATH, encoding="utf-8")]
print(f"dev: {len(dev_sentences)}  test: {len(test_sentences)}")


dev: 1000  test: 1000


In [5]:
def stream_training_corpus(lang, target_size, exclude_set, seed=RANDOM_SEED,
                             max_frac_offset=0.85):
    from huggingface_hub import get_hf_file_metadata, hf_hub_url
    url = hf_hub_url("ai4bharat/IndicCorpV2", f"data/{lang}.txt", repo_type="dataset")
    size = get_hf_file_metadata(url).size
    rng = random.Random(seed)
    offset = rng.randint(0, int(size * max_frac_offset))

    try:
        from indicnlp.tokenize import sentence_tokenize
        def split_sents(p):
            return sentence_tokenize.sentence_split(p, lang=lang)
    except ImportError:
        import re
        _splitter = re.compile(r"(?<=[.?!])\s+")
        def split_sents(p):
            return [s for s in _splitter.split(p) if s]

    TAMIL_BLOCK = (0x0B80, 0x0BFF)
    def tamil_ratio(s):
        n_alpha = sum(1 for ch in s if ch.isalpha())
        if n_alpha == 0:
            return 0.0
        n_tamil = sum(1 for ch in s if TAMIL_BLOCK[0] <= ord(ch) <= TAMIL_BLOCK[1])
        return n_tamil / n_alpha

    def is_clean(s, min_words=3, max_words=120, min_tamil_ratio=0.6):
        if "http://" in s or "https://" in s or "www." in s:
            return False
        words = s.split()
        if not (min_words <= len(words) <= max_words):
            return False
        return tamil_ratio(s) >= min_tamil_ratio

    seen = set()
    pool = []
    headers = {"Range": f"bytes={offset}-"}
    pbar = tqdm(total=target_size, desc="training sentences collected")
    n_excluded = 0
    with requests.get(url, headers=headers, stream=True, timeout=60) as r:
        r.raise_for_status()
        first = True
        for raw in r.iter_lines():
            if first:
                first = False
                continue
            if not raw:
                continue
            try:
                para = raw.decode("utf-8").strip()
            except UnicodeDecodeError:
                continue
            if not para:
                continue
            for s in split_sents(para):
                s = " ".join(s.split())
                if not is_clean(s):
                    continue
                if s in exclude_set:
                    n_excluded += 1
                    continue
                if s in seen:
                    continue
                seen.add(s)
                pool.append(s)
                pbar.update(1)
                if len(pool) >= target_size:
                    pbar.close()
                    print(f"(skipped {n_excluded} sentences that matched dev/test)")
                    return pool
    pbar.close()
    print(f"(skipped {n_excluded} sentences that matched dev/test; "
          f"stream ended early with only {len(pool)}/{target_size})")
    return pool


In [6]:
if DEBUG:
    assert os.path.exists(DEBUG_TRAIN_PATH), (
        "Copy Lab4/data/train_debug_12000.txt into data/ first.")
    train_sentences = [l.rstrip("\n") for l in open(DEBUG_TRAIN_PATH, encoding="utf-8")]
    print(f"[DEBUG] loaded {len(train_sentences)} training sentences from the debug subset")
else:
    if os.path.exists(FULL_TRAIN_PATH):
        train_sentences = [l.rstrip("\n") for l in open(FULL_TRAIN_PATH, encoding="utf-8")]
        print(f"loaded cached {len(train_sentences)} training sentences from {FULL_TRAIN_PATH}")
    else:
        exclude_set = set(dev_sentences) | set(test_sentences)
        t0 = time.time()
        train_sentences = stream_training_corpus(LANG, TARGET_TRAIN_SIZE, exclude_set)
        print(f"streamed {len(train_sentences)} sentences in {time.time()-t0:.1f}s")
        with open(FULL_TRAIN_PATH, "w", encoding="utf-8") as f:
            for s in train_sentences:
                f.write(s + "\n")
        print(f"saved to {FULL_TRAIN_PATH}")


[DEBUG] loaded 12000 training sentences from the debug subset


In [7]:
train_set = set(train_sentences)
overlap_dev = train_set & set(dev_sentences)
overlap_test = train_set & set(test_sentences)
assert len(overlap_dev) == 0, f"LEAKAGE: {len(overlap_dev)} dev sentences found in train!"
assert len(overlap_test) == 0, f"LEAKAGE: {len(overlap_test)} test sentences found in train!"
print(f"Leakage check passed: train={len(train_sentences)}, dev={len(dev_sentences)}, "
      f"test={len(test_sentences)}, 0 overlap")


Leakage check passed: train=12000, dev=1000, test=1000, 0 overlap


## 3. Tokenize

Tamil words in this corpus are whitespace-separated, so tokenization is
simply `sentence.split()` — the same word-level tokenization Lab 4's n-gram
models used (deliberately consistent across labs; Lab 3's BPE/WordPiece
subword tokenizers are a separate, unrelated tokenization used only there).


In [8]:
tokenized_train = [s.split() for s in train_sentences]
tokenized_dev = [s.split() for s in dev_sentences]
tokenized_test = [s.split() for s in test_sentences]

n_tokens = sum(len(t) for t in tokenized_train)
print(f"{len(tokenized_train):,} training sentences, {n_tokens:,} tokens total")


12,000 training sentences, 134,925 tokens total


## 4. Train Word2Vec, FastText, and Doc2Vec

Same `vector_size` / `window` / `min_count` / `epochs` across all three so
comparisons in later sections aren't confounded by mismatched hyperparameters.
`workers=1` trades training speed for a fully reproducible run (see §1).


In [9]:
t0 = time.time()
w2v_model = Word2Vec(
    sentences=tokenized_train,
    vector_size=VECTOR_SIZE, window=WINDOW, min_count=MIN_COUNT,
    workers=WORKERS, epochs=EPOCHS, seed=RANDOM_SEED,
)
print(f"Word2Vec trained in {time.time()-t0:.1f}s -- vocab size = {len(w2v_model.wv):,}")


Word2Vec trained in 0.6s -- vocab size = 4,076


In [10]:
t0 = time.time()
ft_model = FastText(
    sentences=tokenized_train,
    vector_size=VECTOR_SIZE, window=WINDOW, min_count=MIN_COUNT,
    workers=WORKERS, epochs=EPOCHS, seed=RANDOM_SEED,
)
print(f"FastText trained in {time.time()-t0:.1f}s -- vocab size = {len(ft_model.wv):,}")


FastText trained in 3.1s -- vocab size = 4,076


In [11]:
t0 = time.time()
tagged_train = [TaggedDocument(words=toks, tags=[str(i)]) for i, toks in enumerate(tokenized_train)]
d2v_model = Doc2Vec(
    documents=tagged_train,
    vector_size=VECTOR_SIZE, window=WINDOW, min_count=MIN_COUNT,
    workers=WORKERS, epochs=EPOCHS, seed=RANDOM_SEED,
)
print(f"Doc2Vec trained in {time.time()-t0:.1f}s -- vocab size = {len(d2v_model.wv):,}, "
      f"{len(d2v_model.dv):,} document vectors")


Doc2Vec trained in 1.7s -- vocab size = 4,076, 12,000 document vectors


## 5. Sanity check: do the neighbors look real?

Before trusting anything downstream (clusters, nearest-sentence retrieval),
spot-check that `most_similar` returns plausible neighbors, not noise — a
quick, necessary gut check on `min_count`/`epochs` for whatever scale we
just trained at. Also demonstrates FastText's actual practical advantage:
querying a word **never seen during training** works anyway, because
FastText embeds it from character n-grams; the identical call on Word2Vec
raises `KeyError` since Word2Vec has no representation for OOV words at all.


In [12]:
most_common_words = [w for w, _ in sorted(
    ((w, w2v_model.wv.get_vecattr(w, "count")) for w in w2v_model.wv.index_to_key),
    key=lambda x: -x[1])[:5]]

for w in most_common_words:
    print(f"'{w}'  (Word2Vec)  ->", [t for t, _ in w2v_model.wv.most_similar(w, topn=6)])
print()
for w in most_common_words[:3]:
    print(f"'{w}'  (FastText)  ->", [t for t, _ in ft_model.wv.most_similar(w, topn=6)])


'ஒரு'  (Word2Vec)  -> ['நீங்கள்', 'உங்கள்', 'ஆனால்', 'அது', 'அதை', 'சில']
'.'  (Word2Vec)  -> ['எப்படி?', "''", '"', 'More', 'இப்ப', '!']
'மற்றும்'  (Word2Vec)  -> ['உள்ள', 'போன்ற', 'தனது', 'பல', 'புதிய', 'பிற']
'இந்த'  (Word2Vec)  -> ['அவர்', 'என்பதை', 'நாம்', 'சில', 'வேலை', 'ஒரே']
'என்று'  (Word2Vec)  -> ['நான்', 'என்ன', '”', 'ஆனா', 'இப்போ', '?']

'ஒரு'  (FastText)  -> ['என்றால்', 'என்பதால்', 'என்பதையும்', 'இப்போதும்', 'சொல்லும்', 'நாம்']
'.'  (FastText)  -> ["'", '"', '”', "''", '?', 'More']
'மற்றும்'  (FastText)  -> ['நிலையங்கள்', 'ஆண்கள்', 'நாட்கள்', 'ஓட்டங்கள்', 'மக்கள்', ',க்கள்']


In [13]:
# FastText vs Word2Vec on a genuinely out-of-vocabulary string (never seen in
# training -- a nonsense/misspelled token, guaranteed absent from the corpus)
oov_probe = "xyzabc_not_a_real_word_123"

print("Word2Vec on OOV word:")
try:
    w2v_model.wv[oov_probe]
    print("  (unexpectedly found a vector)")
except KeyError as e:
    print(f"  KeyError, as expected -- Word2Vec has no representation for unseen words: {e}")

print("\nFastText on the same OOV word:")
vec = ft_model.wv[oov_probe]
print(f"  succeeded -- vector of shape {vec.shape} synthesized from character n-grams")


Word2Vec on OOV word:
  KeyError, as expected -- Word2Vec has no representation for unseen words: "Key 'xyzabc_not_a_real_word_123' not present"

FastText on the same OOV word:
  succeeded -- vector of shape (100,) synthesized from character n-grams


## 6. K-means clustering of word vectors

Clustering is restricted to the **`TOP_N_WORDS_FOR_CLUSTERING` most frequent**
words in the Word2Vec vocabulary, not the full vocabulary. At full corpus
scale, `min_count=5` still leaves a long tail of rare, often noisy inflected
forms whose vectors are poorly estimated (seen only a handful of times);
including them would make every cluster's "20 closest words" list dominated
by that noise rather than by genuinely coherent groups. Restricting to
frequent words is the standard practical fix and is the single biggest
quality lever here.

`N_CLUSTERS` is not tuned against any held-out metric (there isn't a
natural one for unsupervised word clustering here) — it's set by a simple
rule of thumb (fewer clusters for the smaller debug vocabulary, more for the
full-scale one) and stated explicitly rather than grid-searched.


In [14]:
word_counts = sorted(
    ((w, w2v_model.wv.get_vecattr(w, "count")) for w in w2v_model.wv.index_to_key),
    key=lambda x: -x[1],
)
top_words = [w for w, _ in word_counts[:TOP_N_WORDS_FOR_CLUSTERING]]
X = np.array([w2v_model.wv[w] for w in top_words])
print(f"Clustering {len(top_words):,} words (vector dim={X.shape[1]}) into {N_CLUSTERS} clusters")

kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=RANDOM_SEED, n_init=10)
labels = kmeans.fit_predict(X)


Clustering 300 words (vector dim=100) into 15 clusters


In [15]:
def closest_words_per_cluster(X, top_words, labels, centers, k=WORDS_PER_CLUSTER):
    result = {}
    for c in range(centers.shape[0]):
        idx = np.where(labels == c)[0]
        if len(idx) == 0:
            result[c] = []
            continue
        d = np.linalg.norm(X[idx] - centers[c], axis=1)
        order = idx[np.argsort(d)][:k]
        result[c] = [top_words[i] for i in order]
    return result

cluster_words = closest_words_per_cluster(X, top_words, labels, kmeans.cluster_centers_)

cluster_sizes = pd.Series(labels).value_counts().sort_index()
for c in range(N_CLUSTERS):
    words_str = ", ".join(cluster_words[c])
    print(f"cluster {c:>2} (size={cluster_sizes.get(c, 0):>4}): {words_str}")


cluster  0 (size=   7): தேதி, கடந்த, முதல், ஆண்டு, ஆண்டில், மற்றும், ஆம்
cluster  1 (size=  38): எந்த, போது, பற்றி, விட்டு, தான், என்பதை, இருக்கும், நாம், செய்ய, இன்னும், அவர், அவர்கள், இந்த, வேலை, பெரிய, கொண்டு, வைத்து, சில, மேலும், மிக
cluster  2 (size=   1): மு.
cluster  3 (size=   1): .
cluster  4 (size=  27): நீர், அப்போது, இந்நிலையில், அவன், நான்கு, *, பெற, முதலில், முடியும்., நீண்ட, மேலும்,, நேரம், என்னை, வழக்கு, நடந்து, நாங்கள், வைரஸ், கொள்ள, ஓர், இடங்களில்
cluster  5 (size=  61): இதில், தகவல், அடுத்த, முக்கிய, உடல், வருகிறது., இரு, மூன்று, மட்டுமே, ஐக்கிய, 2, பெரும், 3, சமூக, ஜனாதிபதி, உதவி, நடைபெற்ற, நிலையில், ஒன்று, இரண்டு
cluster  6 (size=   6): எம்., எஸ்., டி., ஐ., பி., தலைவர்
cluster  7 (size=   2): க., தி.
cluster  8 (size=   6): நான், என்ன, அந்த, என், வந்து, என்று
cluster  9 (size=  19): மாநில, முதல்வர், என,, முன்னாள், சார்பில், கட்சி, ஏ., எல்., தமிழக, அமைச்சர், ஜி., சி., சென்னை, மத்திய, பா., அரசு, ஸ்டாலின், -, |
cluster 10 (size=  30): இல்லாமல், இப்போது, எல்லாம், எடுத்

## 7. Sentence vectors: averaged word embeddings vs. Doc2Vec

Three vectorization strategies per sentence:

- **Word2Vec-avg**: mean of each in-vocabulary word's Word2Vec vector
  (out-of-vocabulary words are simply skipped — Word2Vec has no
  representation for them).
- **FastText-avg**: mean of each word's FastText vector — no skipping
  needed, since FastText can embed any string via subword n-grams.
- **Doc2Vec**: `model.infer_vector(tokens)` on the (unseen, held-out)
  sentence. Inference is stochastic by default; `epochs` is passed
  explicitly and the model itself was trained with a fixed seed and
  `workers=1` to keep this as reproducible as gensim allows.


**Preprocessing note added after inspecting the first run's output**:
averaging raw word vectors (punctuation tokens included) produced sentence
vectors that were almost all near-identical — mean pairwise cosine
similarity across 300 random dev sentences was **0.91-0.99** for
Word2Vec-avg/FastText-avg, and the "nearest sentence" for every query came
back with similarity ≈1.000 regardless of actual topic. This is a known
failure mode of naive mean-pooling: extremely frequent tokens (here,
punctuation like `.`, `,`, `?`, and a handful of grammatical particles that
appear in nearly every sentence) inject a large *shared* component into
every sentence's average, so nearly all sentences end up pointing in almost
the same direction in vector space — a "small corpus" version of the
embedding-space *anisotropy* problem documented in the sentence-embedding
literature. The fix applied below, and used consistently for all three
methods (including as input to Doc2Vec's `infer_vector`, so the comparison
stays apples-to-apples): drop punctuation-only tokens before building any
sentence vector. This alone dropped the mean pairwise similarity from ~0.91
to ~0.07 in a controlled check.


In [16]:
import re

def is_word_token(tok):
    """True for tokens containing at least one letter -- filters out pure
    punctuation/symbol tokens (".", ",", "?", quote marks, etc.) that would
    otherwise dominate a mean-pooled sentence vector by sheer frequency."""
    return bool(re.search(r"[^\W\d_]", tok, re.UNICODE))


def sentence_vec_avg(tokens, wv, allow_oov=True):
    vecs = []
    for w in tokens:
        if not is_word_token(w):
            continue
        if allow_oov:
            vecs.append(wv[w])   # FastText: always succeeds
        elif w in wv:
            vecs.append(wv[w])
    if not vecs:
        return np.zeros(wv.vector_size)
    return np.mean(vecs, axis=0)


def build_sentence_vectors(tokenized_sents):
    w2v_vecs = np.array([sentence_vec_avg(t, w2v_model.wv, allow_oov=False) for t in tokenized_sents])
    ft_vecs = np.array([sentence_vec_avg(t, ft_model.wv, allow_oov=True) for t in tokenized_sents])
    d2v_input = [[w for w in t if is_word_token(w)] for t in tokenized_sents]
    d2v_vecs = np.array([d2v_model.infer_vector(t, epochs=EPOCHS) for t in d2v_input])
    return {"Word2Vec-avg": w2v_vecs, "FastText-avg": ft_vecs, "Doc2Vec": d2v_vecs}


dev_vectors = build_sentence_vectors(tokenized_dev)
test_vectors = build_sentence_vectors(tokenized_test)
for name, arr in dev_vectors.items():
    print(f"{name:15s} dev shape={arr.shape}  test shape={test_vectors[name].shape}")


Word2Vec-avg    dev shape=(1000, 100)  test shape=(1000, 100)
FastText-avg    dev shape=(1000, 100)  test shape=(1000, 100)
Doc2Vec         dev shape=(1000, 100)  test shape=(1000, 100)


## 8. Nearest sentence within each set

For every dev sentence, find its most similar **other** dev sentence (cosine
similarity, self excluded); same for test. Done separately for each of the
three vectorization methods, so the retrieved neighbors can be compared.


**Second fix, same root cause**: even after removing punctuation, mean-pooled
(and to a lesser extent Doc2Vec) sentence vectors can still share a common
"average direction" baked in by whatever's structurally frequent across the
whole set (common grammatical patterns, the most frequent content words,
etc.). The standard correction — a simplified version of the postprocessing
step from Arora et al.'s SIF sentence embeddings — is to **mean-center**
each split's vectors (subtract that split's own average vector) before
computing cosine similarity, once per method, so similarity reflects how a
sentence differs from the "typical" sentence in the set rather than how much
of the shared average direction it happens to contain. Applied below to all
three methods for a fair comparison.


In [17]:
def mean_center(vectors):
    return vectors - vectors.mean(axis=0, keepdims=True)


def nearest_within_set(vectors):
    vectors = mean_center(vectors)
    norm = vectors / (np.linalg.norm(vectors, axis=1, keepdims=True) + 1e-10)
    sim = norm @ norm.T
    all_pairwise_mean = float(np.nanmean(np.where(np.eye(len(sim), dtype=bool), np.nan, sim)))
    np.fill_diagonal(sim, -np.inf)
    nn_idx = np.argmax(sim, axis=1)
    nn_sim = sim[np.arange(len(vectors)), nn_idx]
    return nn_idx, nn_sim, all_pairwise_mean


def nearest_summary(vectors_dict, sentences, split_name, n_examples=5):
    rows = []
    nn_results = {}
    for method, vecs in vectors_dict.items():
        nn_idx, nn_sim, all_pairwise_mean = nearest_within_set(vecs)
        nn_results[method] = (nn_idx, nn_sim)
        rows.append({"split": split_name, "method": method,
                      "mean_ALL_pairwise_sim": all_pairwise_mean,
                      "mean_nearest_neighbor_sim": float(nn_sim.mean()),
                      "min_nn_sim": float(nn_sim.min()),
                      "max_nn_sim": float(nn_sim.max())})
    return pd.DataFrame(rows), nn_results


dev_summary_df, dev_nn = nearest_summary(dev_vectors, dev_sentences, "dev")
test_summary_df, test_nn = nearest_summary(test_vectors, test_sentences, "test")
pd.concat([dev_summary_df, test_summary_df], ignore_index=True)


,split,method,mean_ALL_pairwise_sim,mean_nearest_neighbor_sim,min_nn_sim,max_nn_sim
0,dev,Word2Vec-avg,0.009829,0.975773,0.504663,1.000000
1,dev,FastText-avg,0.005973,0.984613,0.649986,1.000000
2,dev,Doc2Vec,-0.000722,0.858533,0.220778,0.997187
3,test,Word2Vec-avg,0.009177,0.972244,0.443781,1.000000
4,test,FastText-avg,0.002091,0.985806,0.617640,1.000000
5,test,Doc2Vec,-0.000127,0.852012,0.190667,0.996140


**Reading this table**: `mean_ALL_pairwise_sim` averages cosine similarity
over *every* pair in the split (~500,000 pairs for 1000 sentences) — this is
close to 0 for all three methods, confirming the mean-centered embedding
space is not degenerate (most sentence pairs are, correctly, not similar).
`mean_nearest_neighbor_sim` instead looks only at each sentence's single
*best* match out of the other 999 — and that stays substantially higher
(~0.85-0.99). This gap is expected, not a bug: picking the best of 999
candidates is a "max of many draws" statistic that comes out high even in a
well-behaved space, and it's further pushed up here by two genuine
properties of this corpus rather than an embedding defect — (1) scraped
Tamil news text contains real near-duplicate/templated phrasing (repeated
byline structures, boilerplate phrasing, sports-score patterns), so a high
best-match score can be a *correct* finding, not an artifact; and (2) at
`DEBUG` scale, Word2Vec/FastText are trained on only ~135,000 tokens —
tiny by the standards these algorithms are normally used at (typically
millions-to-billions of tokens) — so the embeddings are still meaningfully
undertrained, and `mean_nearest_neighbor_sim` should be treated as
provisional until re-run at the full 1,000,000-sentence scale, where far
more training signal should produce better-separated representations.

Look at the printed example pairs below with this in mind: some retrieved
neighbors are genuinely topically related, some are not — a real,
reportable limitation of `DEBUG`-scale training, not something the notebook
should paper over.


In [18]:
def show_example_pairs(sentences, nn_results, n=5, seed=RANDOM_SEED):
    rng = np.random.RandomState(seed)
    sample_idx = rng.choice(len(sentences), size=min(n, len(sentences)), replace=False)
    for method, (nn_idx, nn_sim) in nn_results.items():
        print(f"=== {method} ===")
        for i in sample_idx:
            j = nn_idx[i]
            print(f"  query : {sentences[i][:90]}")
            print(f"  neighbor (sim={nn_sim[i]:.3f}): {sentences[j][:90]}")
            print()

print("### DEV set examples ###\n")
show_example_pairs(dev_sentences, dev_nn)


### DEV set examples ###

=== Word2Vec-avg ===
  query : அதிலும் குறிப்பாக பழம்பெரும் தொகுதியான பொன்னேரி தொகுதியில் 1962-ஆம் ஆண்டில் காங்கிரஸ் கட்ச
  neighbor (sim=0.966): சமய அனுஷ்டானங்களின் பின்னர், அங்கு தர்ம போதனையை செய்த கொலன்னாவே சிறி சுமங்கல தேரர், கடந்த 

  query : அவர் மகள் பிரியாவுக்கும், நம் மகனுக்கும், வர்ற தை மாதம் திருமணம் நடத்திடலாம்,'' என்று, வாச
  neighbor (sim=0.812): இந்த புத்தகத்திற்கான முன்னுரையை தலாய்லாமா எழுதியுள்ளார்.

  query : திறந்த நினைவக இணைப்பிற்காக சன் ஸ்டுடியோ தொகுப்பிகள் மற்றும் கருவிகள் ஓப்பன்MP சிறப்பியல்பு
  neighbor (sim=0.982): ஆகவே கனடா, அமேரிக்கா, இங்கிலாந்து மற்றும் ஐரோப்பிய நாடுகளில் இதன் பயன்பாட்டிற்குத் தடை வித

  query : விளம்பரம் எங்கள் ப்ளாக் ல போட்டுட்டாங்கடா.
  neighbor (sim=0.987): இதில் 11,034 ஆண்கள், 11,215 பெண்கள்.

  query : இந்த நீரிணைப்பகுதி எப்போதும் நிலத்துடன் நிலமாக ஒட்டி இருக்கவில்லை என்றும் இந்த பகுதியில் ச
  neighbor (sim=0.995): இந்தப் பிரிவின் செயற்பாடுகளின் மூலம் சிறைச்சாலைக்குள் இடம் பெறும் சட்ட விரோத செயற்பாடுகளை 

=== 

## 9. Summary

- **Corpus/splits**: identical dev/test to Lab 3/4 (copied verbatim);
  training corpus is Lab 4's debug subset (`DEBUG=True`, used to validate
  every cell end-to-end here) or the leakage-checked, streamed
  1,000,000-sentence corpus (`DEBUG=False`, for the Colab run).
- **Models**: Word2Vec, FastText, and Doc2Vec, trained with matched
  hyperparameters (`vector_size=100, window=5, min_count=5, epochs=10`,
  `workers=1` for reproducibility) — §4, following the three linked gensim
  tutorials. §5 sanity-checks that neighbors look real and that FastText
  (unlike Word2Vec) can embed a genuinely out-of-vocabulary word.
- **Clustering**: K-means over the `TOP_N_WORDS_FOR_CLUSTERING` most
  frequent Word2Vec word vectors only (deliberately excluding the rare-word
  tail), each cluster represented by its 20 nearest words to the centroid —
  §6.
- **Sentence retrieval**: every dev/test sentence vectorized three ways
  (Word2Vec-avg, FastText-avg, Doc2Vec), nearest **other** sentence in the
  same split found by cosine similarity — §7-8. The summary table's
  mean/min/max nearest-neighbor similarity per method is a quick
  quantitative comparison of how "tight" each embedding space is; the
  printed example pairs let you eyeball whether the retrieved neighbor is
  actually semantically/topically related or just a coincidence of common
  words.

### Reproducing the full-scale run (Colab)
1. Upload this notebook plus `data/dev.txt` and `data/test.txt` (and, if you
   already have it from Lab 4, `data/train_1000000.txt`) to Colab.
2. Set `DEBUG = False` in §1.
3. Runtime → Run all. If no cached `train_1000000.txt` is present, the
   corpus is streamed fresh (same leakage-checked method as Lab 4) and
   cached for next time.
